**`enrich_parcels_with_placeslab_fmv2026`**

Attach `US_places-placeslab-fmv2026` fair-market-value predictions and predictors to parcels.

# Configure

In [ ]:
import argparse

from openplaces.api import get_admin
from openplaces.io.enricher import enrich
from openplaces.io.ingester import ingest
from openplaces.recipe import get_output_path, get_recipe_by_id
from openplaces.utils import pretty_print

parser = argparse.ArgumentParser(
    description='Enrich MA parcels with placeslab-fmv2026 fair-market-value evidence'
)
parser.add_argument(
    '--ingest_recipe_id',
    help='Reference parcel recipe to ingest (e.g. "US_parcel-placeslab-fmv2026")',
)
parser.add_argument(
    '--enrich_recipe_id',
    help='Enrichment recipe (e.g. "US_parcel_parcel-placeslab-fmv2026")',
)
parser.add_argument(
    '--entity_recipe_id',
    help='Harmonized entity recipe to enrich (e.g. "US_parcel-spine-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to process (e.g. "US-MA-MI")',
    nargs='+',
    default=None,
)
parser.add_argument(
    '--reprocess',
    help='Reprocess admin IDs even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
# Make a map to look up county codes

get_admin('US-MA', 3, geom=True).explore()

In [ ]:
ARGS_TEST = (
    '--ingest_recipe_id US_parcel-placeslab-fmv2026 '
    '--enrich_recipe_id US_parcel_parcel-placeslab-fmv2026 '
    '--entity_recipe_id US_parcel-spine-2026 '
    '--admin_ids US-PA-DA '
    # Barnstable, Bristol, Essex, Middlesex, Norfolk, Plymouth, Suffolk (Boston), Worcester
    # '--admin_ids US-MA-BA US-MA-BR US-MA-ES US-MA-MI US-MA-NO US-MA-PL US-MA-SU US-MA-WO '
    '--reprocess '
    '--redownload '
    '--verbose '
)

args = parser.parse_args(ARGS_TEST.split())
args

In [ ]:
pretty_print(get_recipe_by_id(args.ingest_recipe_id))
pretty_print(get_recipe_by_id(args.enrich_recipe_id))

# Ingest legacy parcels

Downloads county boundaries, predictions, and predictors (via ``gdown``).

Missing partitions are skipped without failing.

In [ ]:
ingest(
    args.ingest_recipe_id,
    admin_ids=args.admin_ids,
    verbose=args.verbose,
    redownload=args.redownload,
    reprocess=args.reprocess,
)

# Enrich

Link current parcels (`US_parcel-spine-2026`) to imported parcels Enriches current parcels  with area-weighted variables from ``placeslab``

In [ ]:
enrich(
    args.enrich_recipe_id,
    admin_ids=args.admin_ids,
    entity_recipe_id=args.entity_recipe_id,
    reprocess=args.reprocess,
    verbose=args.verbose,
)

---

# Convert to script

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True

convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(ARGS_TEST, committed=COMMIT)

# Inspect results

In [ ]:
from openplaces.io import read_parquet

sample_admin_id = next(a for a in args.admin_ids if a != 'US-MA-SU')  # has boundaries

out_path = get_output_path(
    get_recipe_by_id(args.enrich_recipe_id),
    sample_admin_id,
    entity_recipe_id=get_recipe_by_id(args.entity_recipe_id),
)
evidence = read_parquet(out_path)

In [ ]:
evidence.sample(2).T

## Coverage

In [ ]:
print(f'{len(evidence):,} rows, {len(evidence.columns)} evidence columns')
print((evidence.notna().mean() * 100).round(1))

## Map

In [ ]:
import matplotlib.pyplot as plt

last_admin_id = args.admin_ids[-1]

spine_path = get_output_path(get_recipe_by_id(args.entity_recipe_id), last_admin_id)
spine = read_parquet(spine_path, geom=True)

last_out_path = get_output_path(
    get_recipe_by_id(args.enrich_recipe_id),
    last_admin_id,
    entity_recipe_id=get_recipe_by_id(args.entity_recipe_id),
)
last_evidence = read_parquet(last_out_path)

map_cols = ['elevation_fmv2026', 'land_value_per_ha_log_region_nb_2010_fmv2026']
gdf = spine[['geometry']].join(last_evidence[map_cols])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
titles = ['Elevation (m)', 'Log land value per ha (region-nb, 2010)']
for ax, col, title in zip(axes, map_cols, titles):
    gdf.plot(
        column=col,
        ax=ax,
        cmap='gist_earth' if col.startswith('elevation') else 'RdYlGn_r',
        legend=True,
        missing_kwds={'color': 'lightgrey'},
    )
    ax.set_title(f'{title} — {last_admin_id}')
    ax.set_axis_off()
fig.tight_layout()